In [3]:
%pip install selenium requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 5.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 5.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 3.3 MB/s eta 0:00:00

[notice] A new release of pip available: 22.2.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install undetected-chromedriver

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 985.2 kB/s eta 0:00:00 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.3/173.3 kB 2.9 MB/s eta 0:00:00a 0:00:01
Using legacy 'setup.py install' for undetected-chromedriver, since package 'wheel' is not installed.
  Running setup.py install for undetected-chromedriver ... done

[notice] A new release of pip available: 22.2.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install webdriver-manager


[notice] A new release of pip available: 22.2.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService # Updated import
from webdriver_manager.chrome import ChromeDriverManager # Added import
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.select import Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

import requests
import os
import re
import time

# --- Configuration ---
BASE_URL = "https://engelsiz.kanald.com.tr"
START_PAGE_URL = "https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-ilk-bolum"
SERIES_NAME = "Esref Ruya"
DOWNLOAD_DIR_BASE = "kanald_downloads"

# --- Helper Functions --- (Keep your sanitize_filename and download_file functions as they are)
def sanitize_filename(name):
    """Removes invalid characters for filenames and replaces spaces."""
    name = re.sub(r'[<>:"/\\|?*]', '_', name)
    name = re.sub(r'\s+', '_', name)
    name = name.replace('.', '')
    return name

def download_file(url, local_filepath):
    """Downloads a file from a URL to a local path."""
    os.makedirs(os.path.dirname(local_filepath), exist_ok=True)
    print(f"Downloading: {url}")
    print(f"To: {local_filepath}")
    try:
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(local_filepath, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print(f"Successfully downloaded: {os.path.basename(local_filepath)}")
        return local_filepath
    except requests.exceptions.Timeout:
        print(f"Timeout while downloading {url}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred during download of {url}: {e}")
    return None

# --- Main Scraper Logic ---
def main_scraper():
    # Initialize WebDriver using webdriver-manager
    print("Setting up ChromeDriver using webdriver-manager...")
    driver = None # Initialize driver to None for the finally block
    try:
        driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))
        print("ChromeDriver setup successfully.")
    except Exception as e:
        print(f"Error setting up ChromeDriver with webdriver-manager: {e}")
        return

    wait = WebDriverWait(driver, 20) # Default wait time for elements

    series_folder_name = sanitize_filename(SERIES_NAME)
    series_download_path = os.path.join(DOWNLOAD_DIR_BASE, series_folder_name)
    os.makedirs(series_download_path, exist_ok=True)
    print(f"Downloads will be saved in: {os.path.abspath(series_download_path)}")

    try:
        driver.get(START_PAGE_URL)
        print(f"Navigated to start page: {START_PAGE_URL}")

        episode_options_info = []
        try:
            episode_dropdown_el = wait.until(EC.presence_of_element_located((By.ID, "episodes")))
            episode_select = Select(episode_dropdown_el)
            for option in episode_select.options:
                rel_url = option.get_attribute("value")
                title = option.text.strip()
                if rel_url and title:
                    episode_options_info.append({"url_path": rel_url, "title": title})
            print(f"Found {len(episode_options_info)} episodes.")
        except TimeoutException:
            print("Could not find the episode dropdown (ID: 'episodes'). Exiting.")
            return # No driver.quit() here, finally block will handle it

        if not episode_options_info:
            print("No episodes found in the dropdown. Exiting.")
            return # No driver.quit() here, finally block will handle it

        for episode_info in episode_options_info:
            episode_relative_url = episode_info["url_path"]
            episode_title = episode_info["title"]
            episode_url = BASE_URL + episode_relative_url
            
            sanitized_episode_title = sanitize_filename(episode_title)
            episode_specific_folder = os.path.join(series_download_path, sanitized_episode_title)
            os.makedirs(episode_specific_folder, exist_ok=True)

            print(f"\n--- Processing Episode: {episode_title} ({episode_url}) ---")
            
            try: # Add a try-except block for individual episode processing
                current_base_url = driver.current_url.split('?')[0]
                if current_base_url != episode_url:
                    print(f"Navigating to episode page: {episode_url}")
                    driver.get(episode_url)
                    # More robust wait: wait for body or a common element you expect on all episode pages
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
                    # Specifically wait for the player container if it's crucial for subsequent steps
                    wait.until(EC.presence_of_element_located((By.ID, "huluPlayer"))) 
                    print(f"Successfully navigated to: {episode_url}")


                part_numbers_to_process = ["1"]
                try:
                    parts_dropdown_el = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, "parts")))
                    parts_select = Select(parts_dropdown_el)
                    current_part_options = [opt.get_attribute("value") for opt in parts_select.options if opt.get_attribute("value")]
                    if current_part_options:
                        part_numbers_to_process = current_part_options
                    print(f"Found parts for {episode_title}: {part_numbers_to_process}")
                except TimeoutException:
                    print(f"No 'parts' dropdown found for {episode_title}. Assuming single part.")
                
                for part_num in part_numbers_to_process:
                    print(f"  -- Processing Part {part_num} --")
                    
                    part_specific_url = episode_url + f"?part={part_num}"

                    current_part_page_url = driver.current_url
                    if current_part_page_url != part_specific_url:
                        print(f"  Navigating to part URL: {part_specific_url}")
                        driver.get(part_specific_url)
                        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body"))) # Wait for page load
                        print(f"  Successfully navigated to part URL: {part_specific_url}")

                    try:
                        # This is a critical wait. If it fails, we know the script with var part isn't there.
                        wait.until(EC.presence_of_element_located((By.XPATH, "//script[contains(text(), 'new QuarkPlayer') and contains(text(), 'var part')]")))
                    except TimeoutException:
                        print(f"  Player script with 'var part' did not load for {part_specific_url}. Skipping this part.")
                        # You could try to get page source here for debugging if needed:
                        # print(f"Page source for {part_specific_url}:\n{driver.page_source[:2000]}\n...")
                        continue 
                    
                    time.sleep(1) 

                    media_content_path = None
                    try:
                        all_scripts = driver.find_elements(By.TAG_NAME, "script")
                        for script_element in all_scripts:
                            # script_html = script_element.get_attribute("innerHTML") # Safer way
                            script_html = driver.execute_script("return arguments[0].innerHTML;", script_element)

                            if script_html and "new QuarkPlayer" in script_html and "var part" in script_html:
                                match = re.search(r"var\s+part\s*=\s*'([^']+)'", script_html)
                                if match:
                                    media_content_path = match.group(1)
                                    print(f"  Extracted media content path: {media_content_path}")
                                    break
                        # Fallback if script parsing fails
                        if not media_content_path:
                            try:
                                video_tag = WebDriverWait(driver, 5).until(
                                    EC.presence_of_element_located((By.CSS_SELECTOR, "video#player_1748699022294_html5_api, video.vjs-tech"))
                                )
                                video_src = video_tag.get_attribute("src")
                                if video_src and video_src.startswith("https://cdn.duhnet.tv/"):
                                    media_content_path = video_src.replace("https://cdn.duhnet.tv/", "", 1)
                                    print(f"  Extracted media content path from video tag (fallback): {media_content_path}")
                            except TimeoutException:
                                print(f"  Fallback: Video tag not found or 'src' not in expected format for {episode_title} - Part {part_num}.")
                            except NoSuchElementException: # Should be caught by TimeoutException
                                print(f"  Fallback: Video tag not found for {episode_title} - Part {part_num}.")
                        
                        if not media_content_path: # Check again after fallback
                             print(f"  Could not extract 'var part' from script nor find a direct video source for {episode_title} - Part {part_num}.")
                             continue

                    except Exception as e_extract: # More specific exception for extraction
                        print(f"  Error extracting media path for {episode_title} Part {part_num}: {e_extract}")
                        continue

                    if not media_content_path:
                        print(f"  Failed to find media path. Skipping download for {episode_title} Part {part_num}.")
                        continue

                    # --- Download logic (remains the same) ---
                    video_dl_url = "https://cdn.duhnet.tv/" + media_content_path
                    subtitle_base = os.path.splitext(media_content_path)[0]
                    subtitle_dl_url = "https://cdn.duhnet.tv/" + subtitle_base + ".srt"

                    base_file_name_from_path = os.path.basename(media_content_path)
                    file_stem, file_ext = os.path.splitext(base_file_name_from_path)
                    
                    if len(part_numbers_to_process) > 1 or part_num != "1":
                        local_video_filename = f"{sanitized_episode_title}_Part_{part_num}_{file_stem}{file_ext}"
                        local_subtitle_filename = f"{sanitized_episode_title}_Part_{part_num}_{file_stem}.srt"
                    else:
                        local_video_filename = f"{sanitized_episode_title}_{file_stem}{file_ext}"
                        local_subtitle_filename = f"{sanitized_episode_title}_{file_stem}.srt"

                    video_filepath = os.path.join(episode_specific_folder, local_video_filename)
                    subtitle_filepath = os.path.join(episode_specific_folder, local_subtitle_filename)

                    print(f"  Attempting to download video: {video_dl_url}")
                    download_file(video_dl_url, video_filepath)
                    
                    print(f"  Attempting to download subtitle: {subtitle_dl_url}")
                    download_file(subtitle_dl_url, subtitle_filepath)
                    
                    time.sleep(3) # Be respectful

            # Catch errors specific to processing a single episode/part navigation or initial load
            except TimeoutException as e_timeout:
                print(f"Timeout while processing episode {episode_title} ({episode_url}): {e_timeout}")
                print("Skipping this episode.")
            except Exception as e_episode: # Catch other unexpected errors for this episode
                print(f"An unexpected error occurred while processing episode {episode_title} ({episode_url}): {e_episode}")
                # You could print driver.page_source here if the error is very cryptic
                # print(f"Page source at time of error for {episode_url}:\n{driver.page_source[:2000]}\n...")
                print("Skipping this episode.")
                # Optionally, you can re-raise if you want the whole script to stop
                # raise

    except Exception as e_main: # Catch errors in the main setup or episode iteration loop
        print(f"A critical error occurred in the main scraper: {e_main}")
        import traceback
        traceback.print_exc() # Print full stack trace for critical errors
    finally:
        if driver: # Check if driver was initialized
            print("\nScraping process finished or encountered critical error. Quitting driver.")
            driver.quit()

if __name__ == "__main__":
    main_scraper()

Setting up ChromeDriver using webdriver-manager...
ChromeDriver setup successfully.
Downloads will be saved in: /Users/halilergul/Desktop/personal_projects/voice-data/notebooks/kanald_downloads/Esref_Ruya
Navigated to start page: https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-ilk-bolum
Found 10 episodes.

--- Processing Episode: 10. Bölüm (https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-son-bolum) ---
Navigating to episode page: https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-son-bolum
Successfully navigated to: https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-son-bolum
Found parts for 10. Bölüm: ['1']
  -- Processing Part 1 --
  Navigating to part URL: https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-son-bolum?part=1
  Successfully navigated to part URL: https://engelsiz.kanald.com.tr/esref-ruya/bolumler/esref-ruya-son-bolum?part=1
  Extracted media content path: C13/2013kanald/dizi/esrefruya/er10.mp4
  Attempting to